# LELA60331 Week 12 Seminar: Hidden layers

The classifiers we have built so far, logistic regression and the softmax classifier, divide up the space of inputs with **straight** borders. This week we meet a very simple problem that no straight border can solve, the **exclusive or** (XOR) problem, and see how a network with an extra **hidden layer** solves it. Then we work out, step by step, how such a network is trained.

| Part | What you do |
|---|---|
| 1 | See whether logistic regression can learn AND, OR and XOR |
| 2 | Run a network with a hidden layer that solves XOR |
| 3 | The forward pass through a network, step by step |
| 4 | The backward pass, step by step: **backpropagation** |
| 5 | Train a network on XOR, and look inside it |

Each activity has a cell to complete (replace every `???`) followed by a ✅ **check** cell.

**Lectures:** the network in Part 2 is the one from the *Neural Networks 1* slides, and Parts 3 and 4 follow the worked example in the *Neural Networks 2: Training* lecture step by step.

In [ ]:
# ▶ Run this cell first. It defines the check() helper used after each activity.
# You don't need to understand how it works: it prints ✅ when your answer is right and ❌ when it isn't yet.
def check(label, condition):
    try:
        passed = bool(condition() if callable(condition) else condition)
    except Exception as err:
        passed = False
        label += f"  (error: {type(err).__name__}: {err})"
    print(("✅ " if passed else "❌ ") + label)
    return passed

This cell sets up `dot` (Week 4), `mat_vec` (Week 7) and `sigmoid` (Week 5). Run it.

In [ ]:
import math
import random
import matplotlib.pyplot as plt

def dot(a, b):
    # the dot product of two lists of the same length (Week 4)
    total = 0
    for i in range(len(a)):
        total += a[i] * b[i]
    return total

def mat_vec(matrix, vector):
    # matrix-vector multiplication: the dot product of each row of the matrix with the vector (Week 7)
    result = []
    for row in matrix:
        result.append(dot(row, vector))
    return result

def sigmoid(z):
    return 1 / (1 + math.exp(-z))

---
# Part 1 · AND, OR and XOR

Each of these three functions takes two inputs, each 0 or 1, and gives an output of 0 or 1:

| x1 | x2 | x1 AND x2 | x1 OR x2 | x1 XOR x2 |
|---|---|---|---|---|
| 0 | 0 | 0 | 0 | 0 |
| 0 | 1 | 0 | 1 | 1 |
| 1 | 0 | 0 | 1 | 1 |
| 1 | 1 | 1 | 1 | 0 |

* **AND** is 1 only when both inputs are 1.
* **OR** is 1 when at least one input is 1.
* **XOR** ("exclusive or") is 1 when **exactly one** input is 1.

Can logistic regression learn these? Here are the four inputs, as the rows of a matrix (Week 7), and a function that trains a logistic regression model on them, just as you did in Week 7. It returns the trained weights and bias. Run the cell.

In [ ]:
inputs = [[0, 0],
          [0, 1],
          [1, 0],
          [1, 1]]

def train_logistic(X, y, lr=0.5, n_epochs=5000):
    """Train a logistic regression model, as in Week 7, and return its weights and bias."""
    random.seed(1)
    weights = [random.random() for j in range(len(X[0]))]
    bias = 0
    n = len(y)
    for epoch in range(n_epochs):
        q = [sigmoid(z + bias) for z in mat_vec(X, weights)]
        errors = [q[k] - y[k] for k in range(n)]
        for j in range(len(weights)):
            weights[j] -= lr * dot([row[j] for row in X], errors) / n
        bias -= lr * sum(errors) / n
    return weights, bias

def predict_logistic(X, weights, bias):
    return [round(sigmoid(z + bias), 2) for z in mat_vec(X, weights)]

### Can logistic regression learn AND, OR and XOR?

Here are the targets for AND, OR and XOR, one for each row of `inputs`, made as in the LELA60341 Week 12 workbook. Run the cell. Which functions does logistic regression learn? What does it predict for XOR?

In [ ]:
y_and = [int(x[0] == 1 and x[1] == 1) for x in inputs]
y_or = [int(x[0] == 1 or x[1] == 1) for x in inputs]
y_xor = [int(x[0] != x[1]) for x in inputs]

for name, targets in [("AND", y_and), ("OR", y_or), ("XOR", y_xor)]:
    weights, bias = train_logistic(inputs, targets)
    print(name, " targets:", targets, " probabilities:", predict_logistic(inputs, weights, bias))

The plot below shows why. For AND and OR, a straight line can separate the 1s (circles) from the 0s (crosses). For XOR, the two 1s are in opposite corners, and no single straight line can separate them from the two 0s. Logistic regression gives up, and gives every input a probability of about 0.5.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, (name, targets) in zip(axes, [("AND", [0, 0, 0, 1]), ("OR", [0, 1, 1, 1]), ("XOR", [0, 1, 1, 0])]):
    for row, t in zip(inputs, targets):
        if t == 1:
            ax.scatter(row[0], row[1], marker="o", s=200)
        else:
            ax.scatter(row[0], row[1], marker="x", s=200)
    ax.set_title(name)
    ax.set_xlim(-0.5, 1.5)
    ax.set_ylim(-0.5, 1.5)
plt.show()

---
# Part 2 · A network that solves XOR

A **multilayer network** puts a **hidden layer** of units between the inputs and the output:

```
 inputs (layer 0)        hidden units (layer 1)        output (layer 2)
   x1  ─┐                 ┌─ h1 ─┐
        ├──   W1, b1   ───┤      ├──   w2   ──  output
   x2  ─┘                 └─ h2 ─┘
```

Each hidden unit works just like one class in last week's softmax classifier: it has its own row of weights in a matrix `W1` and its own bias in a list `b1`, so the hidden units' sums are `mat_vec(W1, x)` plus the biases. Each hidden unit then applies an **activation function** to its sum. The output is a weighted sum of the hidden units' values, with the weights `w2`.

We use the **ReLU** activation function ("rectified linear unit"), which turns negative numbers into 0 and leaves positive numbers alone. The activation function is what makes the hidden layer useful: without it, two layers of weighted sums would just add up to one weighted sum, and we would be back to a straight border.

Here is the `relu` function, which you wrote in the LELA60341 Week 12 workbook. Run the cell.

In [ ]:
def relu(z):
    if z > 0:
        return z
    else:
        return 0

print([relu(z) for z in [-2, -0.5, 0, 1.5, 3]])      # [0, 0, 0, 1.5, 3]

Here are weights that solve XOR: the network from the lecture slides. It has two hidden units, so `W1` has two rows, one per hidden unit, and `b1` and `w2` have two numbers each.

In [ ]:
W1 = [[1, 1],       # hidden unit 1's weights for x1 and x2
      [1, 1]]       # hidden unit 2's weights
b1 = [0, -1]        # the biases of hidden units 1 and 2
w2 = [1, -2]        # the output's weights for hidden units 1 and 2

### Activity 1 — A forward pass through the network

Calculating the network's output from an input is called the **forward pass**. For each input `x`:

1. `sums`: the hidden units' sums: `mat_vec` with `W1` and `x`, plus the biases. *Hint:* the loop adds the biases for you.
2. `hidden`: the hidden units' values: `relu` of each sum
3. `output`: the output: the dot product of `w2` with `hidden`

You should find that the outputs are exactly the XOR targets.

In [ ]:
xor_outputs = []
for x in inputs:
    sums = ???                    # 1.
    for j in range(len(sums)):
        sums[j] += b1[j]
    hidden = ???         # 2.
    output = ???                 # 3.
    xor_outputs.append(output)
    print("input:", x, " hidden:", hidden, " output:", output)

In [ ]:
# ✅ Check
check("the outputs are XOR", lambda: xor_outputs == [0, 1, 1, 0])

### Activity 2 — How does it work?

Look at the hidden values you printed. Fill in the two descriptions below, choosing from `"at least one input is 1"`, `"both inputs are 1"` and `"neither input is 1"`. Then explain to your neighbour why `h1 - 2 * h2` gives XOR.

In [ ]:
h1_is_positive_when = ???
h2_is_positive_when = ???

In [ ]:
# ✅ Check
check("h1", lambda: h1_is_positive_when == "at least one input is 1")
check("h2", lambda: h2_is_positive_when == "both inputs are 1")

The first hidden unit computes OR and the second computes AND, both of which logistic regression *can* learn. The output then combines them: "OR, but not AND", which is XOR. The hidden layer has turned the inputs into new features that make the problem easy.

---
# Part 3 · The forward pass, step by step

Of course, we don't want to choose a network's weights by hand: we want it to **learn** them. This part and the next follow the worked example in the *Neural Networks 2: Training* lecture. Here is a network with **three** hidden units and some starting weights, and a single training input `x` whose correct output is 1.

It will help to write the forward pass as a function. `forward(W1, b1, w2, x)` returns **two** values (LELA60341 Week 4): the hidden values and the output, because the backward pass needs both.

In [ ]:
W1 = [[-0.59,  0.34],       # hidden unit 0's weights for x1 and x2
      [ 0.75, -0.17],       # hidden unit 1's weights
      [-0.95,  0.12]]       # hidden unit 2's weights
b1 = [-0.72, -0.60, 0.60]   # the hidden units' biases
w2 = [0.93, -0.37, 0.38]    # the output's weights

x = [0, 1]
target = 1

### Activity 3 — A `forward` function

Complete `forward(W1, b1, w2, x)`, using the same three steps as in Activity 1: the sums, the hidden values, and the output.

In [ ]:
def forward(W1, b1, w2, x):
    sums = ???
    for j in range(len(sums)):
        sums[j] += b1[j]
    hidden = ???
    output = ???
    return hidden, output

hidden, output = forward(W1, b1, w2, x)
print([round(h, 4) for h in hidden], round(output, 4))           # you should get [0, 0, 0.72] and 0.2736

In [ ]:
# ✅ Check
check("hidden", lambda: [round(h, 4) for h in hidden] == [0, 0, 0.72])
check("output", lambda: round(output, 4) == 0.2736)
check("the XOR network still works", lambda: [forward([[1, 1], [1, 1]], [0, -1], [1, -2], x)[1] for x in inputs] == [0, 1, 1, 0])

---
# Part 4 · The backward pass, step by step

The output, 0.27, should be 1. As in linear regression in Week 4, we measure how wrong it is with the difference between the output and the correct value:

`diff = output - target`

Working out how to change each weight, starting from this error at the output and working **backwards** through the network, is called the **backward pass**, or **backpropagation**. We do it in two stages.

### Stage 1: the output's weights

The output is a weighted sum of the hidden values, just like a linear regression whose features are the hidden values. So the gradient for `w2[j]` is exactly as in Week 4: `diff * hidden[j]`. A hidden unit that was more active gets a bigger change; a hidden unit whose value was 0 gets no change at all.

### Activity 4 — The gradients for the output's weights

1. `diff`: the output minus the target
2. `dw2`: the gradient for each of the output's weights: `diff * hidden[j]` for each hidden unit `j`

In [ ]:
diff = ???                                     # 1.
dw2 = ???       # 2.
print(round(diff, 4), [round(g, 4) for g in dw2])             # you should get -0.7264 and [0, 0, -0.523]
# (Python sometimes prints 0 as -0.0: it is the same number)

In [ ]:
# ✅ Check
check("diff", lambda: round(diff, 4) == -0.7264)
check("dw2", lambda: [round(g, 4) for g in dw2] == [0, 0, -0.523])

### Stage 2: the hidden units' weights

How much is each hidden unit to blame for the error? Its share of the blame, its **error**, is:

* `diff`, the output's error, times `w2[j]`, how strongly hidden unit `j` affects the output
* times 1 if the hidden unit was active (`hidden[j] > 0`), and 0 if the ReLU cut it off, because then changing its inputs a little makes no difference to the output

`hidden_error[j] = diff * w2[j] * (1 if hidden[j] > 0, otherwise 0)`

Once we have each hidden unit's error, its weights and bias are just like those of logistic regression or of one class in a softmax classifier (last week): the gradient for `W1[j][i]` is `hidden_error[j] * x[i]`, and the gradient for `b1[j]` is `hidden_error[j]`.

### Activity 5 — The hidden units' errors and gradients

1. complete the loop that makes `hidden_error`: for each hidden unit `j`, if it was active, its error is `diff * w2[j]`; otherwise it is 0
2. `dW1`: complete the inner loop, so that row `j` holds `hidden_error[j] * x[i]` for each input `i` (compare the gradient matrix of a softmax classifier, last week)
3. `db1`: the gradients for the hidden units' biases

In [ ]:
hidden_error = []
for j in range(len(hidden)):
    if hidden[j] > 0:
        hidden_error.append(???)            # 1.
    else:
        hidden_error.append(???)

dW1 = []
for j in range(len(hidden)):
    row = []
    for i in range(len(x)):
        row.append(???)          # 2.
    dW1.append(row)
db1 = ???                            # 3.

print([round(e, 4) for e in hidden_error])            # you should get [0, 0, -0.276]
print([[round(g, 4) for g in row] for row in dW1])    # you should get [[0, 0], [0, 0], [0, -0.276]]

In [ ]:
# ✅ Check
check("hidden_error", lambda: [round(e, 4) for e in hidden_error] == [0, 0, -0.276])
check("dW1", lambda: [[round(g, 4) for g in row] for row in dW1] == [[0, 0], [0, 0], [0, -0.276]])
check("db1", lambda: [round(g, 4) for g in db1] == [0, 0, -0.276])

### Activity 6 — The update

Update every weight and bias, with a learning rate of 0.2: **new value = old value − learning rate × gradient**.

1. `w2`: the output's new weights
2. `W1`: complete the inner loop to update the hidden units' weights
3. `b1`: the hidden units' new biases

Which weights change, and which stay the same? Why? Does the output get closer to 1?

In [ ]:
lr = 0.2
w2 = ???             # 1.
for j in range(len(W1)):
    for i in range(len(W1[j])):
        W1[j][i] = ???                    # 2.
b1 = ???             # 3.

print([[round(v, 3) for v in row] for row in W1])     # you should get [[-0.59, 0.34], [0.75, -0.17], [-0.95, 0.175]]
print([round(v, 3) for v in b1])                      # you should get [-0.72, -0.6, 0.655]
print([round(v, 3) for v in w2])                      # you should get [0.93, -0.37, 0.485]
print("output before: 0.2736, after:", round(forward(W1, b1, w2, x)[1], 4))

In [ ]:
# ✅ Check
check("W1", lambda: [[round(v, 3) for v in row] for row in W1] == [[-0.59, 0.34], [0.75, -0.17], [-0.95, 0.175]])
check("b1", lambda: [round(v, 3) for v in b1] == [-0.72, -0.6, 0.655])
check("w2", lambda: [round(v, 3) for v in w2] == [0.93, -0.37, 0.485])
check("the output got closer to 1", lambda: forward(W1, b1, w2, x)[1] > 0.2736)

Only the weights connected to hidden unit 2, the only active one, changed. The other hidden units were cut off by the ReLU, so they didn't affect the output, and they get no share of the blame.

---
# Part 5 · Training a network on XOR

Now we put the forward pass, the backward pass and the update in a loop, and train a network with three hidden units on the four XOR examples. This network updates its weights after **each** example in turn, rather than once per pass through all the data.

To keep the loop short, here are the backward pass and the update from Part 4, as a function. It returns the new weights and biases. Read through it: every line is one you have written. Run the cell.

In [ ]:
def backward_and_update(W1, b1, w2, x, target, lr):
    hidden, output = forward(W1, b1, w2, x)
    diff = output - target
    # stage 1: the output's weights
    dw2 = [diff * hidden[j] for j in range(len(hidden))]
    # stage 2: the hidden units' weights and biases
    hidden_error = []
    for j in range(len(hidden)):
        if hidden[j] > 0:
            hidden_error.append(diff * w2[j])
        else:
            hidden_error.append(0)
    new_w2 = [w2[j] - lr * dw2[j] for j in range(len(w2))]
    new_W1 = [[W1[j][i] - lr * hidden_error[j] * x[i] for i in range(len(x))] for j in range(len(W1))]
    new_b1 = [b1[j] - lr * hidden_error[j] for j in range(len(b1))]
    return new_W1, new_b1, new_w2

### Activity 7 — The training loop

Each epoch goes through the four examples. For each example:

1. add its squared difference, `(output - target) ** 2`, to `total_error`. *Hint:* use `forward` to get the output.
2. update the weights and biases with `backward_and_update`

The cell after it shows the trained network's output for each input, which should be close to 0, 1, 1, 0.

In [ ]:
xor_targets = [0, 1, 1, 0]
hidden_size = 3
lr = 0.35
n_epochs = 1000

random.seed(1)
W1 = [[random.random(), random.random()] for j in range(hidden_size)]
b1 = [random.random() for j in range(hidden_size)]
w2 = [random.random() for j in range(hidden_size)]

loss_history = []
for epoch in range(n_epochs):
    total_error = 0
    for k in range(len(inputs)):
        x = inputs[k]
        target = xor_targets[k]
        hidden, output = ???
        total_error += ???                              # 1.
        W1, b1, w2 = ???        # 2.
    loss_history.append(total_error)

plt.plot(loss_history)
plt.xlabel("epoch")
plt.ylabel("loss")
plt.show()

for x in inputs:
    print(x, round(forward(W1, b1, w2, x)[1], 3))

In [ ]:
# ✅ Check
check("the loss went down", lambda: loss_history[-1] < loss_history[0])
check("the network has learned XOR", lambda: [round(forward(W1, b1, w2, x)[1], 2) for x in inputs] == [0, 1, 1, 0])

### Activity 8 — Looking inside the trained network

1. `trained_hidden`: the hidden values for each of the four inputs (a list of four lists). *Hint:* `forward` returns the hidden values first.
2. `active`: for each input, the list of the hidden units that are active (whose value is greater than 0)

Print the trained weights too. Which hidden units are active for which inputs? Can you explain in words how this network computes XOR, as you did in Activity 2?

In [ ]:
trained_hidden = ???
active = ???
for k in range(len(inputs)):
    print(inputs[k], [round(h, 2) for h in trained_hidden[k]], active[k])
print("W1:", [[round(v, 2) for v in row] for row in W1])
print("b1:", [round(v, 2) for v in b1])
print("w2:", [round(v, 2) for v in w2])

In [ ]:
# ✅ Check
check("trained_hidden", lambda: len(trained_hidden) == 4 and trained_hidden[1] == forward(W1, b1, w2, inputs[1])[0])
check("active", lambda: active == [[j for j in range(hidden_size) if h[j] > 0] for h in trained_hidden])

*Write your explanation here.*

⭐ **Challenge (optional):** change `random.seed(1)` to `random.seed(5)` and train again. What has gone wrong? Look at the hidden values. *Hint:* what does the ReLU do to a hidden unit whose sum is negative for every input, and what does that mean for its share of the blame in the backward pass?

---
## ✏️ Summary

You have:

* seen that logistic regression can learn AND and OR, but not XOR, because no straight border separates XOR's classes
* used a **hidden layer** with the **ReLU** activation function to solve XOR: the hidden units make new features from the inputs
* carried out the **forward pass** (sums, ReLU, output) and the **backward pass** (the output's error, the hidden units' share of it, the gradients) through a network
* trained a network on XOR, and looked inside it to see what its hidden units have learned